# Trigger `parquet_hms_registration` DAG — End-to-End Test Harness

One-shot setup + trigger + verification for **DAG 4 (Parquet HMS Registration)**. The DAG takes
*plain parquet files on S3* (no Hive metadata — what a prior data-only migration leaves behind),
infers the schema, creates an `EXTERNAL` Hive table on top, and registers Hive-style
`key=value` partitions via `MSCK REPAIR TABLE`. Run cells top-to-bottom on a JupyterHub kernel
whose Spark session shares the same Hive Metastore the DAG writes to.

### What this notebook does

| Step | Action |
|---|---|
| 0 | Configuration (Airflow REST API, S3 surface, target HMS DB, tracking/report locations) |
| 1 | Helpers — Spark session, S3 FS helpers, Airflow JWT/REST helpers |
| 2 | Seed plain parquet test data on S3 covering every layout the DAG must handle |
| 3 | Drop the target HMS database so the DAG registers fresh (not `SKIPPED`) |
| 4 | Build the minimal `database / table / s3_location` Excel config (plus rows that must be skipped) and upload it to S3 |
| 5 | Trigger a `parquet_hms_registration` run via the Airflow REST API |
| 6 | Poll the run to a terminal state and print the task summary |
| 7 | Read both tracking tables for this run |
| 8 | **Assertions** — verify the hardened behaviors (STRING partitions, `mergeSchema`, empty-partition counting, parse-skip) |
| 9 | Render the DAG's HTML report inline |
| 10 | Cleanup (drop test DB, wipe seeded S3, delete this run's tracking rows) |

### Scenarios seeded in Step 2

| Table | Layout | Exercises |
|---|---|---|
| `sales_partitioned` | partitioned by `dt` | basic single-column partition + `MSCK` |
| `events_multi` | partitioned by `region`/`month` (month = `01`,`02`) | multi-level discovery; STRING partition typing keeps the leading zero |
| `customers_flat` | unpartitioned | no `PARTITIONED BY`, no `MSCK`, no partition validation |
| `evolving_flat` | two files, schemas `(id,name)` then `(id,name,email)` | `mergeSchema=true` unions footers — `email` must survive |
| `logs_empty_part` | partitioned by `dt`, plus a 0-row-parquet partition and a truly-empty (`_SUCCESS`-only) partition | leaf-dir partition counting incl. empty dirs that `MSCK` registers |

### Prerequisites

- JupyterHub Spark session with Iceberg + S3 (`s3a://`) access (NX1 default), sharing the DAG's Hive Metastore.
- Airflow webserver reachable from JH, with credentials allowed to trigger DAGs. Set `AIRFLOW_BASE_URL`, `AIRFLOW_USERNAME`, `AIRFLOW_PASSWORD`.
- `S3_BUCKET` and `S3_TENANT_PREFIX` env vars for the bucket/prefix the DAG's Spark session can read.

In [ ]:
# ── Step 0 — Configuration ───────────────────────────────────────────────────────
import os

# --- Airflow REST API (in-cluster URL bypasses Keycloak; FAB auth -> JWT via /auth/token) ---
AIRFLOW_BASE_URL = os.environ.get("AIRFLOW_BASE_URL", "")
AIRFLOW_USERNAME = os.environ.get("AIRFLOW_USERNAME", "")
AIRFLOW_PASSWORD = os.environ.get("AIRFLOW_PASSWORD", "")
DAG_ID           = "parquet_hms_registration"

# --- S3 surface (must be readable/writable by the DAG's Spark session via s3a://) ---
# TENANT_PREFIX example: f"s3a://{BUCKET}/<tenant>"
BUCKET        = os.environ.get("S3_BUCKET", "")
TENANT_PREFIX = f"s3a://{BUCKET}/es-tenant-2"
TEST_BASE     = f"{TENANT_PREFIX}/parquet_hms_e2e_test"
EXCEL_S3_PATH = f"{TENANT_PREFIX}/configs/parquet_hms_e2e_test.xlsx"

# --- HMS target: the DAG creates this database; this notebook drops it between runs ---
TARGET_DB = "hms_e2e_db"

# --- Tracking + report (must match the DAG's get_config(): tracking_database / report_output_location) ---
TRACKING_DB     = "migration_tracking"
REPORT_LOCATION = f"{TENANT_PREFIX}/migration_reports"

print("Configuration loaded.")
print(f"  Airflow URL    : {AIRFLOW_BASE_URL or '(unset — set AIRFLOW_BASE_URL)'}")
print(f"  Airflow user   : {AIRFLOW_USERNAME or '(unset — set AIRFLOW_USERNAME/PASSWORD)'}")
print(f"  DAG ID         : {DAG_ID}")
print(f"  Test S3 base   : {TEST_BASE}")
print(f"  Excel S3 path  : {EXCEL_S3_PATH}")
print(f"  Target HMS DB  : {TARGET_DB}")
print(f"  Tracking DB    : {TRACKING_DB}")
print(f"  Report location: {REPORT_LOCATION}")

## Step 1 — Helpers

Spark session, a few S3 FileSystem helpers (delete / mkdirs / recursive tree print), and the
Airflow JWT + REST helpers (same pattern as `trigger_mapr_to_s3_dag_e2e_test.ipynb`).

In [ ]:
import requests
from py4j.java_gateway import java_import
from pyspark.sql import SparkSession

try:
    _ = spark
    print(f"Using existing Spark session (version {spark.version})")
except NameError:
    spark = SparkSession.builder \
        .appName("trigger-parquet-hms-e2e") \
        .enableHiveSupport() \
        .getOrCreate()
    print(f"Created Spark session (version {spark.version})")

spark.sparkContext.setLogLevel("WARN")
java_import(spark._jvm, "org.apache.hadoop.fs.*")


def _fs(path):
    return spark._jvm.org.apache.hadoop.fs.FileSystem.get(
        spark._jvm.java.net.URI(path), spark._jsc.hadoopConfiguration()
    )


def s3_delete(path):
    fs = _fs(path)
    p = spark._jvm.org.apache.hadoop.fs.Path(path)
    if fs.exists(p):
        fs.delete(p, True)
        print(f"  deleted: {path}")
    else:
        print(f"  skip (not present): {path}")


def s3_mkdirs(path):
    _fs(path).mkdirs(spark._jvm.org.apache.hadoop.fs.Path(path))


def s3_tree(path, indent=""):
    fs = _fs(path)
    p = spark._jvm.org.apache.hadoop.fs.Path(path)
    if not fs.exists(p):
        print(f"{indent}(missing) {path}")
        return
    for st in sorted(fs.listStatus(p), key=lambda s: s.getPath().getName()):
        name = st.getPath().getName()
        if st.isDirectory():
            print(f"{indent}{name}/")
            s3_tree(st.getPath().toString(), indent + "  ")
        else:
            print(f"{indent}{name}  ({st.getLen()} B)")


_AIRFLOW_TOKEN = {"value": None}


def _airflow_token():
    if _AIRFLOW_TOKEN["value"]:
        return _AIRFLOW_TOKEN["value"]
    if not AIRFLOW_USERNAME or not AIRFLOW_PASSWORD:
        raise RuntimeError("Set AIRFLOW_USERNAME and AIRFLOW_PASSWORD before calling Airflow.")
    url = AIRFLOW_BASE_URL.rstrip("/") + "/auth/token"
    resp = requests.post(url, json={"username": AIRFLOW_USERNAME, "password": AIRFLOW_PASSWORD}, timeout=30)
    if not resp.ok:
        raise RuntimeError(f"POST {url} -> {resp.status_code}: {resp.text[:300]}")
    _AIRFLOW_TOKEN["value"] = resp.json()["access_token"]
    return _AIRFLOW_TOKEN["value"]


def airflow_request(method, path, **kwargs):
    headers = {"Authorization": f"Bearer {_airflow_token()}", **kwargs.pop("headers", {})}
    url = AIRFLOW_BASE_URL.rstrip("/") + path
    resp = requests.request(method, url, headers=headers, timeout=30, **kwargs)
    if resp.status_code == 401:
        _AIRFLOW_TOKEN["value"] = None
        headers["Authorization"] = f"Bearer {_airflow_token()}"
        resp = requests.request(method, url, headers=headers, timeout=30, **kwargs)
    if not resp.ok:
        raise RuntimeError(f"{method} {url} -> {resp.status_code}: {resp.text[:500]}")
    return resp.json() if resp.content else {}


print("Helpers ready.")

## Step 2 — Seed plain parquet test data on S3

Writes parquet directly (no Hive table is created here — that is the DAG's job). `partitionBy`
produces Hive-style `key=value` directories. `month` and `dt` are written as **STRING** columns
so the directory names keep their literal form (`month=01`, not `month=1`).

The `logs_empty_part` table gets two empty-partition variants on top of its two data partitions:

- `dt=2026-06-03` — a **0-row parquet file** (directory with a real but empty parquet). `MSCK REPAIR`
  registers it but a `DISTINCT`-over-data partition count would miss it — the exact case the DAG's
  leaf-directory counting was hardened to handle.
- `dt=2026-06-04` — a **truly empty** directory holding only a `_SUCCESS` marker, no parquet at all.
  `MSCK` still registers a key=value dir with no data files, so the metastore partition count and the
  FS leaf-dir walk agree (4 partitions) while only 2 hold rows — Step 8 asserts this.

In [ ]:
s3_delete(TEST_BASE)

# 1. sales_partitioned — single partition column, dt as STRING
spark.createDataFrame(
    [(1, 10.0, "2026-01-01"), (2, 20.0, "2026-01-01"), (3, 30.0, "2026-01-02")],
    "id INT, amount DOUBLE, dt STRING",
).write.mode("overwrite").partitionBy("dt").parquet(f"{TEST_BASE}/sales_partitioned")

# 2. events_multi — two partition levels; month keeps its leading zero only if typed STRING
spark.createDataFrame(
    [
        (101, "EU", "01", "p1"), (102, "EU", "02", "p2"),
        (103, "US", "01", "p3"), (104, "US", "02", "p4"),
    ],
    "event_id INT, region STRING, month STRING, payload STRING",
).write.mode("overwrite").partitionBy("region", "month").parquet(f"{TEST_BASE}/events_multi")

# 3. customers_flat — unpartitioned
spark.createDataFrame(
    [(1, "Ann", "LV"), (2, "Bo", "EE"), (3, "Cy", "LT")],
    "id INT, name STRING, country STRING",
).write.mode("overwrite").parquet(f"{TEST_BASE}/customers_flat")

# 4. evolving_flat — second file adds an `email` column; only mergeSchema keeps it
spark.createDataFrame([(1, "a"), (2, "b")], "id INT, name STRING") \
    .write.mode("overwrite").parquet(f"{TEST_BASE}/evolving_flat")
spark.createDataFrame([(3, "c", "c@x"), (4, "d", "d@x")], "id INT, name STRING, email STRING") \
    .write.mode("append").parquet(f"{TEST_BASE}/evolving_flat")

# 5. logs_empty_part — two data partitions + two empty-partition variants:
#    dt=2026-06-03 holds a 0-row parquet file; dt=2026-06-04 holds only a _SUCCESS marker (no parquet).
spark.createDataFrame(
    [(1, "boot", "2026-06-01"), (2, "warn", "2026-06-01"), (3, "err", "2026-06-02")],
    "id INT, msg STRING, dt STRING",
).write.mode("overwrite").partitionBy("dt").parquet(f"{TEST_BASE}/logs_empty_part")
spark.createDataFrame([], "id INT, msg STRING") \
    .coalesce(1).write.mode("overwrite").parquet(f"{TEST_BASE}/logs_empty_part/dt=2026-06-03")
_marker = f"{TEST_BASE}/logs_empty_part/dt=2026-06-04/_SUCCESS"
_out = _fs(_marker).create(spark._jvm.org.apache.hadoop.fs.Path(_marker), True)
_out.close()

print("\nSeeded layout:")
s3_tree(TEST_BASE)

## Step 3 — Drop the target HMS database (pre-run reset)

The DAG records a table that already exists as `SKIPPED` (same location → idempotent `MSCK`;
different location → skip). To test a clean registration every run, drop the whole target
database first. This removes only the HMS metadata — the seeded parquet on S3 is external and
untouched.

In [ ]:
spark.sql(f"DROP DATABASE IF EXISTS {TARGET_DB} CASCADE")
print(f"Dropped HMS database (if it existed): {TARGET_DB}")

## Step 4 — Build and upload the Excel config

Three columns — `database`, `table`, `s3_location` — one row per table. The last three rows are
**intentionally invalid** and must be skipped by `parse_parquet_hms_excel` (logged, not failed),
so the finalized run should report exactly **5** tables.

| database | table | s3_location | expected |
|---|---|---|---|
| `hms_e2e_db` | `sales_partitioned` | …/sales_partitioned | registered |
| `hms_e2e_db` | `events_multi` | …/events_multi | registered |
| `hms_e2e_db` | `customers_flat` | …/customers_flat | registered |
| `hms_e2e_db` | `evolving_flat` | …/evolving_flat | registered |
| `hms_e2e_db` | `logs_empty_part` | …/logs_empty_part | registered |
| `hms_e2e_db` | `missing_loc` | _(blank)_ | skipped — missing required cell |
| `hms_e2e_db` | `bad-name` | …/customers_flat | skipped — invalid identifier |
| `hms_e2e_db` | `sales_partitioned` | …/sales_partitioned | skipped — duplicate row |

In [ ]:
from io import BytesIO

import pandas as pd

rows = [
    {"database": TARGET_DB, "table": "sales_partitioned", "s3_location": f"{TEST_BASE}/sales_partitioned"},
    {"database": TARGET_DB, "table": "events_multi",      "s3_location": f"{TEST_BASE}/events_multi"},
    {"database": TARGET_DB, "table": "customers_flat",    "s3_location": f"{TEST_BASE}/customers_flat"},
    {"database": TARGET_DB, "table": "evolving_flat",     "s3_location": f"{TEST_BASE}/evolving_flat"},
    {"database": TARGET_DB, "table": "logs_empty_part",   "s3_location": f"{TEST_BASE}/logs_empty_part"},
    # --- rows below must be skipped at parse time ---
    {"database": TARGET_DB, "table": "missing_loc",       "s3_location": ""},
    {"database": TARGET_DB, "table": "bad-name",          "s3_location": f"{TEST_BASE}/customers_flat"},
    {"database": TARGET_DB, "table": "sales_partitioned", "s3_location": f"{TEST_BASE}/sales_partitioned"},
]
df = pd.DataFrame(rows)

buf = BytesIO()
df.to_excel(buf, index=False, engine="openpyxl")
xlsx_bytes = buf.getvalue()

p = spark._jvm.org.apache.hadoop.fs.Path(EXCEL_S3_PATH)
out = _fs(EXCEL_S3_PATH).create(p, True)
try:
    out.write(xlsx_bytes)
finally:
    out.close()

print(f"Uploaded {len(xlsx_bytes)} bytes to {EXCEL_S3_PATH}")
df

## Step 5 — Trigger a DAG run

POSTs to `/api/v2/dags/parquet_hms_registration/dagRuns` with the Excel path as a conf override.

In [ ]:
from datetime import datetime, timezone

logical_date = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S+00:00")
run_id = f"manual_hms_e2e_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"

trigger_body = {
    "dag_run_id": run_id,
    "logical_date": logical_date,
    "conf": {"excel_file_path": EXCEL_S3_PATH},
}

resp = airflow_request("POST", f"/api/v2/dags/{DAG_ID}/dagRuns", json=trigger_body)
DAG_RUN_ID = resp["dag_run_id"]
print(f"Triggered: dag_run_id={DAG_RUN_ID} state={resp.get('state')}")
print(f"  excel_file_path = {EXCEL_S3_PATH}")

## Step 6 — Monitor the run

Polls every 15s until the run reaches a terminal state, then prints per-task state. Mapped tasks
(`register_parquet_tables`, `validate_registered_tables`, …) appear once per Excel table.

In [ ]:
import time

TERMINAL = {"success", "failed"}
POLL_INTERVAL = 15
TIMEOUT_SECS = 60 * 20

deadline = time.time() + TIMEOUT_SECS
last_state = None
while time.time() < deadline:
    run = airflow_request("GET", f"/api/v2/dags/{DAG_ID}/dagRuns/{DAG_RUN_ID}")
    state = run.get("state")
    if state != last_state:
        print(f"[{time.strftime('%H:%M:%S')}] state = {state}")
        last_state = state
    if state in TERMINAL:
        break
    time.sleep(POLL_INTERVAL)
else:
    print(f"Timed out after {TIMEOUT_SECS}s waiting for terminal state.")

tasks = airflow_request("GET", f"/api/v2/dags/{DAG_ID}/dagRuns/{DAG_RUN_ID}/taskInstances")
print("\nTask summary:")
print(f"  {'task_id':<32} {'map':>4} {'state':<10} {'try':<4} duration")
print(f"  {'-'*32} {'-'*4} {'-'*10} {'-'*4} {'-'*10}")
for t in sorted(tasks.get("task_instances", []), key=lambda x: (x.get("task_id") or "", x.get("map_index", -1))):
    mi = t.get("map_index", -1)
    mi = "" if mi is None or mi < 0 else mi
    print(f"  {t.get('task_id',''):<32} {str(mi):>4} {str(t.get('state','')):<10} {str(t.get('try_number','')):<4} {t.get('duration')}")

## Step 7 — Read the tracking tables for this run

`create_hms_registration_run` stamps the Airflow `run_id` as `dag_run_id`, so we map our
triggered `DAG_RUN_ID` back to the DAG's internal `RUN_ID` (`hms_reg_…`) and pull the per-table
status rows.

In [ ]:
run_rows = spark.sql(
    "SELECT run_id, status, total_tables, successful_tables, failed_tables, skipped_tables "
    f"FROM {TRACKING_DB}.hms_registration_runs "
    f"WHERE dag_run_id = '{DAG_RUN_ID}' "
    "ORDER BY started_at DESC"
).collect()
assert run_rows, f"No hms_registration_runs row for dag_run_id={DAG_RUN_ID}"
RUN_ID = run_rows[0]["run_id"]
print("Run:", run_rows[0].asDict())

status_pd = spark.sql(
    "SELECT database_name, table_name, status, partition_columns, "
    "hms_row_count, parquet_row_count, row_count_match, "
    "hms_partition_count, s3_partition_count, partition_count_match, "
    "validation_status, error_message "
    f"FROM {TRACKING_DB}.hms_registration_status "
    f"WHERE run_id = '{RUN_ID}' "
    "ORDER BY table_name"
).toPandas()
status_by_table = {r["table_name"]: r for _, r in status_pd.iterrows()}
status_pd

## Step 8 — Assertions

Verifies the behaviors that make this DAG correct, not just that it ran — all five tables are
asserted strictly.

- **Run rollup**: status `COMPLETED`, `total_tables == 5` (the 3 bad Excel rows were skipped at parse), zero failures.
- **Per table** (all five): each reaches `VALIDATED` with matching row and partition counts.
- **STRING partition typing**: `events_multi.region` / `.month` register as `string`, and `SHOW PARTITIONS` keeps `month=01` (no leading-zero collapse).
- **Schema evolution**: `evolving_flat` exposes `email` (so `mergeSchema=true` unioned the footers).
- **Unpartitioned**: `customers_flat` has no partition columns.
- **Empty partitions** (`logs_empty_part`): `MSCK` registers all **4** `dt=` dirs — including the 0-row-parquet `dt=2026-06-03` and the `_SUCCESS`-only `dt=2026-06-04` — so `hms_partition_count == s3_partition_count == 4` while only **3** rows across **2** data-bearing partitions exist. The old `DISTINCT`-over-data count (**2**) is asserted to undercount the 4 leaf dirs — exactly the mismatch the leaf-dir counting was hardened to prevent.

The per-partition breakdown is printed for visibility before the assertions run.

In [ ]:
def describe_types(db, tbl):
    types = {}
    for r in spark.sql(f"DESCRIBE {db}.{tbl}").collect():
        col = (r["col_name"] or "").strip()
        if not col or col.startswith("#"):
            continue
        types.setdefault(col, (r["data_type"] or "").strip())
    return types


# All five tables are asserted strictly. logs_empty_part is included: MSCK registers even the
# _SUCCESS-only dt=2026-06-04 dir, so the leaf-dir walk and the metastore agree (4 partitions)
# while only 3 rows across 2 data-bearing partitions exist.
STRICT_TABLES = ("sales_partitioned", "events_multi", "customers_flat", "evolving_flat", "logs_empty_part")

# logs_empty_part is deterministic given the seed: dt=2026-06-01..04, two of them empty.
EXPECTED_LOG_PARTITIONS = 4
EXPECTED_LOG_ROWS = 3
EXPECTED_LOG_DISTINCT_DT = 2  # only the two partitions that hold rows

failures = []
run = run_rows[0]

if run["status"] != "COMPLETED":
    failures.append(f"run status {run['status']!r} != COMPLETED")
if run["total_tables"] != 5:
    failures.append(f"total_tables {run['total_tables']} != 5 (bad Excel rows should be skipped at parse)")
if run["failed_tables"] != 0:
    failures.append(f"failed_tables {run['failed_tables']} != 0")

for tbl in STRICT_TABLES:
    row = status_by_table.get(tbl)
    if row is None:
        failures.append(f"{tbl}: no tracking row")
        continue
    if row["status"] != "VALIDATED":
        failures.append(f"{tbl}: status {row['status']!r} != VALIDATED ({row['error_message']})")
    if not bool(row["row_count_match"]):
        failures.append(f"{tbl}: row mismatch hms={row['hms_row_count']} parquet={row['parquet_row_count']}")
    if not bool(row["partition_count_match"]):
        failures.append(f"{tbl}: partition mismatch hms={row['hms_partition_count']} s3={row['s3_partition_count']}")

# STRING partition typing + leading-zero preserved
ev_types = describe_types(TARGET_DB, "events_multi")
for col in ("region", "month"):
    if ev_types.get(col) != "string":
        failures.append(f"events_multi.{col} type {ev_types.get(col)!r} != string")
ev_parts = [r[0] for r in spark.sql(f"SHOW PARTITIONS {TARGET_DB}.events_multi").collect()]
if not any("month=01" in p for p in ev_parts):
    failures.append(f"events_multi: leading-zero partition month=01 not preserved: {ev_parts}")

# mergeSchema union
evolving_cols = set(describe_types(TARGET_DB, "evolving_flat"))
if "email" not in evolving_cols:
    failures.append(f"evolving_flat missing 'email' (mergeSchema not applied): {sorted(evolving_cols)}")

# unpartitioned
if (status_by_table["customers_flat"]["partition_columns"] or "") != "":
    failures.append(f"customers_flat should be unpartitioned, got {status_by_table['customers_flat']['partition_columns']!r}")

# ── empty partition dirs: MSCK registers them, leaf-dir walk matches, DISTINCT-over-data undercounts ──
print("=== logs_empty_part ===")
lp_loc = f"{TEST_BASE}/logs_empty_part"
lp_fs = _fs(lp_loc)
for st in sorted(lp_fs.listStatus(spark._jvm.org.apache.hadoop.fs.Path(lp_loc)), key=lambda s: s.getPath().getName()):
    name = st.getPath().getName()
    if not st.isDirectory() or "=" not in name:
        continue
    children = [c.getPath().getName() for c in lp_fs.listStatus(st.getPath())]
    n_parquet = sum(1 for c in children if c.endswith(".parquet"))
    kind = f"{n_parquet} parquet file(s)" if n_parquet else f"no parquet ({', '.join(children) or 'empty'})"
    print(f"  {name}: {kind}")

lp = status_by_table["logs_empty_part"]
hms_parts = [r[0] for r in spark.sql(f"SHOW PARTITIONS {TARGET_DB}.logs_empty_part").collect()]
old_distinct = spark.read.parquet(lp_loc).select("dt").distinct().count()
print(f"\nMSCK registered {len(hms_parts)} partitions: {sorted(hms_parts)}")
print(f"DAG tracking : hms_partition_count={lp['hms_partition_count']}  "
      f"s3_partition_count={lp['s3_partition_count']}  partition_count_match={lp['partition_count_match']}")
print(f"row counts   : hms={lp['hms_row_count']}  parquet={lp['parquet_row_count']}  match={lp['row_count_match']}")
print(f"old DISTINCT-over-data dt count: {old_distinct} (vs {EXPECTED_LOG_PARTITIONS} leaf dirs)")

if int(lp["hms_partition_count"]) != EXPECTED_LOG_PARTITIONS:
    failures.append(f"logs_empty_part: hms_partition_count {lp['hms_partition_count']} != {EXPECTED_LOG_PARTITIONS} "
                    f"(MSCK should register both empty partition dirs)")
if int(lp["s3_partition_count"]) != EXPECTED_LOG_PARTITIONS:
    failures.append(f"logs_empty_part: s3_partition_count {lp['s3_partition_count']} != {EXPECTED_LOG_PARTITIONS}")
if "dt=2026-06-04" not in hms_parts:
    failures.append(f"logs_empty_part: truly-empty dt=2026-06-04 not registered by MSCK: {sorted(hms_parts)}")
if int(lp["hms_row_count"]) != EXPECTED_LOG_ROWS:
    failures.append(f"logs_empty_part: hms_row_count {lp['hms_row_count']} != {EXPECTED_LOG_ROWS}")
if old_distinct != EXPECTED_LOG_DISTINCT_DT:
    failures.append(
        f"logs_empty_part: DISTINCT-over-data dt {old_distinct} != {EXPECTED_LOG_DISTINCT_DT} "
        f"(should undercount the {EXPECTED_LOG_PARTITIONS} leaf dirs — the mismatch leaf-dir counting prevents)"
    )

print()
if failures:
    print("\n".join(f"  FAIL: {f}" for f in failures))
    raise AssertionError(f"{len(failures)} strict verification(s) failed")
print("ALL STRICT VERIFICATIONS PASSED")

## Step 9 — Render the HTML report

`generate_hms_html_report` writes `{REPORT_LOCATION}/{RUN_ID}_hms_report.html`. Read it back from
S3 and display inline.

In [ ]:
from IPython.display import HTML, display

report_path = f"{REPORT_LOCATION}/{RUN_ID}_hms_report.html"
print(f"Reading report: {report_path}\n")

p = spark._jvm.org.apache.hadoop.fs.Path(report_path)
fs = _fs(report_path)
if not fs.exists(p):
    print(f"Report not found at {report_path} — REPORT_LOCATION may differ from the DAG's report_output_location.")
else:
    reader = spark._jvm.java.io.BufferedReader(
        spark._jvm.java.io.InputStreamReader(fs.open(p), "UTF-8")
    )
    lines = []
    line = reader.readLine()
    while line is not None:
        lines.append(line)
        line = reader.readLine()
    reader.close()
    display(HTML("\n".join(lines)))

## Step 10 — Cleanup

Drops the test HMS database, wipes the seeded parquet and the Excel config on S3, removes the
report HTML, and deletes this run's rows from both tracking tables. Safe to re-run the whole
notebook from the top afterwards.

In [ ]:
print("Cleaning up test artifacts...\n")

spark.sql(f"DROP DATABASE IF EXISTS {TARGET_DB} CASCADE")
print(f"  dropped HMS database: {TARGET_DB}")

s3_delete(TEST_BASE)
s3_delete(EXCEL_S3_PATH)
try:
    s3_delete(f"{REPORT_LOCATION}/{RUN_ID}_hms_report.html")
except Exception as e:
    print(f"  report delete skipped: {e}")

try:
    spark.sql(f"DELETE FROM {TRACKING_DB}.hms_registration_status WHERE run_id = '{RUN_ID}'")
    spark.sql(f"DELETE FROM {TRACKING_DB}.hms_registration_runs   WHERE run_id = '{RUN_ID}'")
    print(f"  deleted tracking rows for run_id={RUN_ID}")
except Exception as e:
    print(f"  tracking cleanup skipped: {e}")

print("\nCleanup complete.")